# Wash Empire Pixal3D Prototype Assets

Run this in a GPU Colab runtime. Upload one reference PNG per prop, generate GLBs with Pixal3D, then download a zip that drops into `WashEmpireWeb/public/prototype-assets`.

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

print(sys.version)
print(subprocess.check_output(['nvidia-smi'], text=True))

ROOT = Path('/content')
WORK = ROOT / 'wash-empire-pixal3d'
INPUT_DIR = WORK / 'inputs'
OUTPUT_DIR = WORK / 'prototype-assets'
ASSET_DIR = OUTPUT_DIR / 'assets'
INPUT_DIR.mkdir(parents=True, exist_ok=True)
ASSET_DIR.mkdir(parents=True, exist_ok=True)

## Install Pixal3D

Pixal3D's demo wheels are built for Linux/CUDA and may require a Python 3.10 Colab runtime. If this cell fails on a wheel tag, switch to a Python 3.10 runtime or use a Colab image that supports cp310 wheels.

In [ ]:
%cd /content
!rm -rf Pixal3D
!git clone --depth 1 https://github.com/TencentARC/Pixal3D.git
%cd /content/Pixal3D
!python -m pip install --upgrade pip
!python -m pip install -r requirements-hfdemo.txt
!python -m pip install https://github.com/LDYang694/Storages/releases/download/20260430/utils3d-0.0.2-py3-none-any.whl

## Upload Reference Images

Use clean, centered, single-object PNGs. Names should match the target asset ids below.

In [ ]:
from google.colab import files

uploaded = files.upload()
for name, data in uploaded.items():
    target = INPUT_DIR / name
    target.write_bytes(data)
    print('uploaded', target)

In [ ]:
ASSET_SPECS = [
    {
        'id': 'change-vending-alcove',
        'input': 'change-vending-alcove.png',
        'position': [-6.8, 0, 2.1],
        'rotation': [0, 1.5708, 0],
        'scale': [1.05, 1.05, 1.05],
        'fov': 0.32,
        'seed': 42,
    },
    {
        'id': 'self-serve-control-box',
        'input': 'self-serve-control-box.png',
        'position': [-5.32, 0, -2.35],
        'rotation': [0, 0, 0],
        'scale': [0.6, 0.6, 0.6],
        'fov': 0.26,
        'seed': 43,
    },
    {
        'id': 'vacuum-island',
        'input': 'vacuum-island.png',
        'position': [7.4, 0, 2.9],
        'rotation': [0, -0.35, 0],
        'scale': [0.95, 0.95, 0.95],
        'fov': 0.34,
        'seed': 44,
    },
    {
        'id': 'wash-wand-hose',
        'input': 'wash-wand-hose.png',
        'position': [-4.2, 0, -1.8],
        'rotation': [0, 0, 0],
        'scale': [0.75, 0.75, 0.75],
        'fov': 0.22,
        'seed': 45,
    },
]

for spec in ASSET_SPECS:
    print(spec['id'], 'input exists:', (INPUT_DIR / spec['input']).exists())

## Generate GLBs

In [ ]:
%cd /content/Pixal3D

generated = []
for spec in ASSET_SPECS:
    image_path = INPUT_DIR / spec['input']
    if not image_path.exists():
        print('SKIP missing input:', image_path)
        continue

    output_path = ASSET_DIR / f"{spec['id']}.glb"
    cmd = [
        sys.executable,
        'inference.py',
        '--image', str(image_path),
        '--output', str(output_path),
        '--seed', str(spec['seed']),
        '--fov', str(spec['fov']),
    ]
    print('RUN', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    generated.append(spec)

print('generated', [spec['id'] for spec in generated])

## Build WashEmpire Manifest and Download Zip

In [ ]:
manifest = {
    'assets': [
        {
            'id': spec['id'],
            'url': f"/prototype-assets/assets/{spec['id']}.glb",
            'position': spec['position'],
            'rotation': spec['rotation'],
            'scale': spec['scale'],
            'visible': True,
        }
        for spec in generated
    ]
}

(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
zip_base = ROOT / 'wash-empire-pixal3d-assets'
if zip_base.with_suffix('.zip').exists():
    zip_base.with_suffix('.zip').unlink()
shutil.make_archive(str(zip_base), 'zip', OUTPUT_DIR)
print(zip_base.with_suffix('.zip'))
files.download(str(zip_base.with_suffix('.zip')))